# Multi-Agent Library Robot Navigation (Actor–Critic RL + PSO)
**AI Planning & Robotics coursework, The British University in Egypt. Author: Ziad Ayman**

Four robots learn to navigate a simulated library (walls, shelves, moving cleaner robots) with independent Actor–Critic policies.
They share critic values through communication weights tuned by Particle Swarm Optimisation.

> Requirements: `tensorflow`, `pygame`, `numpy`. Part 2 opens a desktop window, so run it locally rather than on a headless server.

## Part 1: Environment, agents, training and PSO optimisation
Defines the Pygame library environment, the `Robot`/`Cleaner` classes, the per-agent Actor–Critic networks (TensorFlow/Keras), inter-agent communication, and the two-stage Particle Swarm Optimisation (communication weights, then hyperparameters). Running it trains the agents and saves their weights.

In [14]:
import pygame
import numpy as np
import math
import random
import time
import json
import tensorflow as tf
from tensorflow.keras import layers, optimizers

pygame.init()

# -------------------- Display / env sizes & flags --------------------
SCREEN_WIDTH = 1500
SCREEN_HEIGHT = 750
wall_thickness = 12

HEADLESS = True
MAX_STEPS = 500

ACTION_SIZE = 5
STATE_SIZE = 8
NUM_AGENTS = 4  # All 4 robots are now trainable

SCREEN = None
if not HEADLESS:
    SCREEN = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("Multi-Agent Library Environment")

# Colors
WHITE = (255,255,255)
RED = (255,0,0)
table_color = (100,255,100)
chair_color = (150,75,0)
shelf_color = (139,69,19)
bg_color = (210,210,210)
wall_color = (100,100,100)
robot_colors = [(50,100,150), (150,50,100), (100,150,50), (150,100,50)]  # Different color per robot
cleaner_color = (230,0,0)

# -------------------- Static obstacles --------------------
static_obstacles = {
    'walls': [
        pygame.Rect(0, 50, SCREEN_WIDTH, wall_thickness),
        pygame.Rect(0, 738, SCREEN_WIDTH, wall_thickness),
        pygame.Rect(0, 50, 12, 700),
        pygame.Rect(1488, 50, 12, 700),
        pygame.Rect(750, 62, 12, 480),
    ],
    'shelves': [
        pygame.Rect(232, 200, 20, 250),
        pygame.Rect(482, 200, 20, 250),
    ],
    'tables': [
        pygame.Rect(972, 200, 75, 40),
        pygame.Rect(1212, 200, 75, 40),
        pygame.Rect(972, 400, 75, 40),
        pygame.Rect(1212, 400, 75, 40),
    ],
    'chairs': [
        pygame.Rect(987, 180, 15, 15),
        pygame.Rect(1017, 180, 15, 15),
        pygame.Rect(987, 245, 15, 15),
        pygame.Rect(1017, 245, 15, 15),
        pygame.Rect(1227, 180, 15, 15),
        pygame.Rect(1257, 180, 15, 15),
        pygame.Rect(1227, 245, 15, 15),
        pygame.Rect(1257, 245, 15, 15),
        pygame.Rect(987, 380, 15, 15),
        pygame.Rect(1017, 380, 15, 15),
        pygame.Rect(987, 445, 15, 15),
        pygame.Rect(1017, 445, 15, 15),
        pygame.Rect(1227, 380, 15, 15),
        pygame.Rect(1257, 380, 15, 15),
        pygame.Rect(1227, 445, 15, 15),
        pygame.Rect(1257, 445, 15, 15),
    ]
}

def check_collision_with_static_obstacles(rect):
    for obstacles_group in static_obstacles.values():
        for obstacle in obstacles_group:
            if rect.colliderect(obstacle):
                return True
    return False

# -------------------- Robot & Cleaner classes --------------------
class Robot:
    num = 0
    width = 20
    height = 50
    max_velocity = 8.0
    min_velocity = 1.0
    all_robots = []

    def __init__(self, position, theta, robot_id):
        Robot.num += 1
        self.num = Robot.num
        self.robot_id = robot_id  # Index for multi-agent
        self.position = np.array(position, dtype=float)
        self.theta = float(theta)
        self.rect = pygame.Rect(int(self.position[0]), int(self.position[1]), Robot.width, Robot.height)
        self.current_velocity = 3.0
        Robot.all_robots.append(self)

    def check_collision_with_other_robots(self, rect):
        for robot in Robot.all_robots:
            if robot.num == self.num:
                continue
            if rect.colliderect(robot.rect):
                return True
        return False

    def apply_action(self, action):
        rot_step = 0.08
        accel = 0.6
        if action == 0:
            self.current_velocity = min(self.current_velocity + accel, Robot.max_velocity)
        elif action == 1:
            self.current_velocity = max(self.current_velocity - accel, -Robot.max_velocity/2.0)
        elif action == 2:
            self.theta = (self.theta - rot_step) % (2*math.pi)
        elif action == 3:
            self.theta = (self.theta + rot_step) % (2*math.pi)
        elif action == 4:
            self.current_velocity *= 0.9

    def update_physics(self):
        future_position = np.zeros(2, dtype=float)
        future_position[0] = self.position[0] + self.current_velocity * math.cos(self.theta)
        future_position[1] = self.position[1] + self.current_velocity * math.sin(self.theta)
        future_rect = pygame.Rect(int(future_position[0]), int(future_position[1]), Robot.width, Robot.height)
        if check_collision_with_static_obstacles(future_rect) or self.check_collision_with_other_robots(future_rect):
            self.theta = (self.theta + math.pi/10) % (2*math.pi)
            self.current_velocity = 0.0
        else:
            self.position = future_position
            self.rect = future_rect

class Cleaner:
    count_cleaners = 0
    max_velocity = 4
    min_velocity = 2
    width = 20
    height = 20
    all_cleaners = []

    def __init__(self, position, theta):
        Cleaner.count_cleaners += 1
        self.num = Cleaner.count_cleaners
        self.position = np.array(position, dtype=float)
        self.theta = float(theta)
        self.current_velocity = Cleaner.min_velocity
        self.rect = pygame.Rect(int(position[0]), int(position[1]), Cleaner.width, Cleaner.height)
        Cleaner.all_cleaners.append(self)

    def check_collision_cleaner(self, future_rect):
        for cleaner in Cleaner.all_cleaners:
            if cleaner.num == self.num:
                continue
            if future_rect.colliderect(cleaner.rect):
                return True
        return False

    def check_collision_robot(self, future_rect):
        for robot in Robot.all_robots:
            if future_rect.colliderect(robot.rect):
                return True
        return False

    def move_cleaner(self):
        future_position = np.zeros(2, dtype=float)
        future_position[0] = self.position[0] + self.current_velocity * math.cos(self.theta)
        future_position[1] = self.position[1] + self.current_velocity * math.sin(self.theta)
        future_rect = pygame.Rect(int(future_position[0]), int(future_position[1]), Cleaner.width, Cleaner.height)
        if (check_collision_with_static_obstacles(future_rect)
            or self.check_collision_cleaner(future_rect)
            or self.check_collision_robot(future_rect)):
            self.theta = (self.theta + math.pi) % (2*math.pi)
        else:
            self.position = future_position
            self.rect = future_rect
            self.current_velocity = Cleaner.min_velocity if self.position[0] >= 750 else Cleaner.max_velocity

# -------------------- Draw --------------------
def draw_environment(screen):
    screen.fill(bg_color)
    font_color = (0, 0, 0)
    font = pygame.font.SysFont('Arial', 26, bold=True)
    screen.blit(font.render("Bookshelves Room", True, font_color), (wall_thickness + 245, 15))
    screen.blit(font.render("Quiet Area", True, font_color), (500 + wall_thickness + 540, 15))
    
    for obstacles_group in static_obstacles.keys():
        if obstacles_group == 'walls':
            obstacle_color = wall_color
        elif obstacles_group == 'shelves':
            obstacle_color = shelf_color
        elif obstacles_group == 'tables':
            obstacle_color = table_color
        elif obstacles_group == 'chairs':
            obstacle_color = chair_color
        for obstacle in static_obstacles[obstacles_group]:
            pygame.draw.rect(screen, obstacle_color, obstacle)

    for robot in Robot.all_robots:
        color = robot_colors[robot.robot_id % len(robot_colors)]
        pygame.draw.rect(screen, color, robot.rect)

    for cleaner in Cleaner.all_cleaners:
        pygame.draw.rect(screen, cleaner_color, cleaner.rect)
        cleaner.move_cleaner()

# -------------------- Env API --------------------
_current_step = 0

def env_seed(s=None):
    random.seed(s)
    np.random.seed(s)

def env_reset(seed=None):
    global _current_step, SCREEN
    env_seed(seed)
    _current_step = 0
    Robot.all_robots = []
    Cleaner.all_cleaners = []
    Robot.num = 0
    Cleaner.count_cleaners = 0
    
    # Initialize all 4 robots with unique IDs
    Robot(np.array([100,500]), 0, robot_id=0)
    Robot(np.array([600,150]), 0, robot_id=1)
    Robot(np.array([880,300]), 0, robot_id=2)
    Robot(np.array([1400,600]), 0, robot_id=3)
    
    Cleaner(np.array([200,500]), 0)
    Cleaner(np.array([900,680]), 0)
    
    if not HEADLESS and SCREEN is None:
        pygame.display.init()
    
    # Return observations for all agents
    return [env_get_obs(i).astype(np.float32) for i in range(NUM_AGENTS)]

def env_get_obs(robot_idx=0):
    robot = Robot.all_robots[robot_idx]
    x_norm = (robot.position[0] / SCREEN_WIDTH) * 2 - 1
    y_norm = (robot.position[1] / SCREEN_HEIGHT) * 2 - 1
    cth = math.cos(robot.theta)
    sth = math.sin(robot.theta)
    
    if Cleaner.all_cleaners:
        dists = [np.linalg.norm(cl.position - robot.position) for cl in Cleaner.all_cleaners]
        idx = int(np.argmin(dists))
        dx = (Cleaner.all_cleaners[idx].position[0] - robot.position[0]) / SCREEN_WIDTH
        dy = (Cleaner.all_cleaners[idx].position[1] - robot.position[1]) / SCREEN_HEIGHT
    else:
        dx, dy = 0.0, 0.0
    
    min_dist = 1e9
    for group in static_obstacles.values():
        for obs in group:
            center = np.array([obs.centerx, obs.centery], dtype=float)
            d = np.linalg.norm(center - robot.position)
            if d < min_dist:
                min_dist = d
    
    if min_dist == 1e9:
        min_dist_norm = 1.0
    else:
        max_diag = math.hypot(SCREEN_WIDTH, SCREEN_HEIGHT)
        min_dist_norm = np.clip(min_dist / max_diag, 0.0, 1.0)
        min_dist_norm = min_dist_norm * 2 - 1
    
    vel_norm = (robot.current_velocity - Robot.min_velocity) / max(1e-8, (Robot.max_velocity - Robot.min_velocity))
    vel_norm = vel_norm * 2 - 1
    
    obs = np.array([x_norm, y_norm, cth, sth, dx, dy, min_dist_norm, vel_norm], dtype=np.float32)
    return np.clip(obs, -1.0, 1.0)

def env_step(actions):
    """
    actions: list of NUM_AGENTS actions
    returns: (observations, rewards, dones, infos) for all agents
    """
    global _current_step
    _current_step += 1
    
    goal_x = 750.0
    observations = []
    rewards = []
    dones = []
    infos = []
    
    # Apply actions and update physics for all robots
    for i, robot in enumerate(Robot.all_robots):
        prev_dist = abs(goal_x - robot.position[0])
        robot.apply_action(actions[i])
        robot.update_physics()
        
        new_dist = abs(goal_x - robot.position[0])
        shaping_reward = 0.5 * (prev_dist - new_dist)
        step_penalty = -0.001
        reward = shaping_reward + step_penalty
        done = False
        
        # Check collisions
        if robot.check_collision_with_other_robots(robot.rect) or check_collision_with_static_obstacles(robot.rect):
            reward = -1.0
            done = True
        
        for cl in Cleaner.all_cleaners:
            if robot.rect.colliderect(cl.rect):
                reward = -1.0
                done = True
        
        if robot.position[0] >= goal_x:
            reward = 10.0
            done = True
        
        if _current_step >= MAX_STEPS:
            done = True
        
        rewards.append(float(reward))
        dones.append(bool(done))
        infos.append({})
    
    # Move cleaners
    for cl in Cleaner.all_cleaners:
        cl.move_cleaner()
    
    # Get observations for all agents
    for i in range(NUM_AGENTS):
        observations.append(env_get_obs(i).astype(np.float32))
    
    return observations, rewards, dones, infos

def env_render():
    if HEADLESS:
        return
    draw_environment(SCREEN)
    pygame.display.flip()

# -------------------- Actor-Critic Model --------------------
class ActorCritic(tf.keras.Model):
    def __init__(self, state_size, action_size, hidden_size=128):
        super(ActorCritic, self).__init__()
        self.fc1 = layers.Dense(hidden_size, activation='relu')
        self.actor = layers.Dense(action_size, activation='softmax')
        self.critic = layers.Dense(1)
    
    def call(self, x):
        x = self.fc1(x)
        return self.actor(x), self.critic(x)

def compute_returns(rewards, gamma):
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = np.array(returns, dtype=np.float32)
    if returns.std() < 1e-8:
        return returns - returns.mean()
    return (returns - returns.mean()) / (returns.std() + 1e-8)

# -------------------- Communication Functions --------------------
def apply_communication(critic_values_list, comm_vector, beta=0.1):
    """
    Apply communication between agents by sharing critic value information.
    critic_values_list: list of critic values for each agent [agent_i_values, ...]
    comm_vector: weights for communication [w0, w1, w2, w3] (normalized)
    beta: communication strength
    """
    new_values = []
    for i in range(NUM_AGENTS):
        values_i = critic_values_list[i].copy()
        msg = np.zeros_like(values_i)
        
        for j in range(NUM_AGENTS):
            if j != i:
                # Weighted message from agent j to agent i
                msg += comm_vector[j] * (critic_values_list[j] - values_i)
        
        # Update values with communication
        new_values.append(values_i + beta * msg)
    
    return new_values

# -------------------- Multi-Agent Training --------------------
def train_multi_agent(comm_vector, max_episodes=300, learning_rate=1e-3, gamma=0.99, 
                      hidden_size=128, entropy_coef=1e-3, beta_comm=0.1, 
                      print_interval=10, headless=True, clip_grad_norm=0.5, 
                      seed=None, verbose=True):
    """
    Train all NUM_AGENTS robots with communication using comm_vector.
    Returns: (avg_total_reward, success_rate, models_list)
    """
    if seed is not None:
        env_seed(seed)
        tf.random.set_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
    
    global HEADLESS
    old_headless = HEADLESS
    HEADLESS = headless
    
    # Create one model per agent
    models = [ActorCritic(STATE_SIZE, ACTION_SIZE, hidden_size) for _ in range(NUM_AGENTS)]
    optimizers_list = [optimizers.Adam(learning_rate) for _ in range(NUM_AGENTS)]
    
    # Build weights
    for model in models:
        model(tf.zeros((1, STATE_SIZE), dtype=tf.float32))
    
    episode_returns = []
    success_counts = [0] * NUM_AGENTS
    
    for ep in range(1, max_episodes + 1):
        states_all = env_reset()
        
        # Storage for each agent
        trajectories = [{'states': [], 'actions': [], 'rewards': []} for _ in range(NUM_AGENTS)]
        dones_all = [False] * NUM_AGENTS
        total_rewards = [0.0] * NUM_AGENTS
        
        while not all(dones_all):
            actions = []
            
            # Select actions for all agents
            for i in range(NUM_AGENTS):
                if not dones_all[i]:
                    s = states_all[i].astype(np.float32)
                    trajectories[i]['states'].append(s)
                    
                    probs, _ = models[i](tf.convert_to_tensor([s]))
                    probs = probs.numpy()[0]
                    action = np.random.choice(ACTION_SIZE, p=probs)
                    trajectories[i]['actions'].append(int(action))
                    actions.append(action)
                else:
                    actions.append(4)  # No-op for done agents
            
            # Step environment
            next_states, rewards, dones, _ = env_step(actions)
            
            # Store rewards and update states
            for i in range(NUM_AGENTS):
                if not dones_all[i]:
                    trajectories[i]['rewards'].append(rewards[i])
                    total_rewards[i] += rewards[i]
                    
                    if dones[i]:
                        dones_all[i] = True
                        if any(r == 10.0 for r in trajectories[i]['rewards']):
                            success_counts[i] += 1
            
            states_all = next_states
        
        episode_returns.append(sum(total_rewards))
        
        # Update networks with communication
        all_critic_values = []
        
        # First pass: compute critic values for all agents
        for i in range(NUM_AGENTS):
            if len(trajectories[i]['states']) == 0:
                all_critic_values.append(np.array([]))
                continue
                
            states_t = tf.convert_to_tensor(np.vstack(trajectories[i]['states']), dtype=tf.float32)
            _, values = models[i](states_t)
            values_np = values.numpy()
            if values_np.ndim > 1:
                values_np = values_np.flatten()
            all_critic_values.append(values_np)
        
        # Apply communication (share critic information)
        if len([v for v in all_critic_values if len(v) > 0]) > 0:
            max_len = max(len(v) for v in all_critic_values if len(v) > 0)
            padded_values = []
            for v in all_critic_values:
                if len(v) == 0:
                    padded_values.append(np.zeros(max_len))
                elif len(v) < max_len:
                    padded_values.append(np.pad(v, (0, max_len - len(v)), 'edge'))
                else:
                    padded_values.append(v)
            
            communicated_values = apply_communication(padded_values, comm_vector, beta_comm)
        else:
            communicated_values = all_critic_values
        
        # Second pass: update each agent's network
        for i in range(NUM_AGENTS):
            if len(trajectories[i]['states']) == 0:
                continue
            
            returns = compute_returns(trajectories[i]['rewards'], gamma)
            states_t = tf.convert_to_tensor(np.vstack(trajectories[i]['states']), dtype=tf.float32)
            actions_t = tf.convert_to_tensor(trajectories[i]['actions'], dtype=tf.int32)
            returns_t = tf.convert_to_tensor(returns, dtype=tf.float32)
            
            with tf.GradientTape() as tape:
                probs_batch, values = models[i](states_t)
                
                # Handle values shape properly
                if values.shape.ndims > 1:
                    values = tf.squeeze(values, axis=-1)
                else:
                    values = tf.reshape(values, [-1])
                
                # Use communicated values for advantage computation
                comm_values = communicated_values[i][:len(trajectories[i]['states'])]
                comm_values_t = tf.convert_to_tensor(comm_values, dtype=tf.float32)
                
                action_masks = tf.one_hot(actions_t, ACTION_SIZE)
                log_probs = tf.math.log(tf.reduce_sum(probs_batch * action_masks, axis=1) + 1e-8)
                
                advantages = returns_t - comm_values_t
                adv_mean = tf.reduce_mean(advantages)
                adv_std = tf.math.reduce_std(advantages) + 1e-8
                advantages = (advantages - adv_mean) / adv_std
                
                actor_loss = -tf.reduce_mean(log_probs * advantages)
                critic_loss = tf.reduce_mean(tf.square(returns_t - values))
                entropy = -tf.reduce_mean(tf.reduce_sum(probs_batch * tf.math.log(probs_batch + 1e-8), axis=1))
                
                loss = actor_loss + 0.5 * critic_loss - entropy_coef * entropy
            
            grads = tape.gradient(loss, models[i].trainable_variables)
            grads, _ = tf.clip_by_global_norm(grads, clip_grad_norm)
            optimizers_list[i].apply_gradients(zip(grads, models[i].trainable_variables))
        
        if verbose and (ep % print_interval == 0 or ep == 1 or ep == max_episodes):
            avg_reward = sum(total_rewards) / NUM_AGENTS
            print(f"Episode {ep}  |  Avg Reward: {avg_reward:.2f}  |  Total: {sum(total_rewards):.2f}")
    
    HEADLESS = old_headless
    avg_return = float(np.mean(episode_returns)) if episode_returns else 0.0
    success_rate = sum(success_counts) / (max_episodes * NUM_AGENTS)
    
    return avg_return, success_rate, models

# -------------------- PSO for Communication Weights --------------------
def pso_optimize_comm(n_particles=6, iters=5, eval_episodes=20, seed=None, verbose=True):
    """
    PSO to find best communication weights for multi-agent learning.
    Returns: (best_comm_vector, best_score)
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    
    # Initialize swarm: each particle is a communication vector
    swarm = np.random.rand(n_particles, NUM_AGENTS)
    swarm = swarm / swarm.sum(axis=1, keepdims=True)  # Normalize
    velocities = np.zeros_like(swarm)
    pbest = swarm.copy()
    pbest_scores = np.full(n_particles, -np.inf)
    
    if verbose:
        print("\n===== INITIAL COMM PSO EVALUATION =====")
    
    # Initial evaluation
    for i in range(n_particles):
        score, _, _ = train_multi_agent(pbest[i], max_episodes=eval_episodes, 
                                        learning_rate=1e-3, gamma=0.99, 
                                        hidden_size=128, entropy_coef=1e-3,
                                        beta_comm=0.1, headless=True, 
                                        seed=seed, verbose=False)
        pbest_scores[i] = score
        if verbose:
            print(f"Particle {i+1}: Score={score:.2f}, Weights={pbest[i]}")
    
    gbest_idx = np.argmax(pbest_scores)
    gbest = pbest[gbest_idx].copy()
    gbest_score = pbest_scores[gbest_idx]
    
    w, c1, c2 = 0.5, 1.5, 1.5
    
    for it in range(iters):
        if verbose:
            print(f"\n===== COMM PSO ITER {it+1}/{iters} | Best={gbest_score:.2f} =====")
        
        for i in range(n_particles):
            r1, r2 = np.random.rand(), np.random.rand()
            velocities[i] = (
                w * velocities[i] +
                c1 * r1 * (pbest[i] - swarm[i]) +
                c2 * r2 * (gbest - swarm[i])
            )
            swarm[i] += velocities[i]
            swarm[i] = np.clip(swarm[i], 0.001, 1)
            swarm[i] = swarm[i] / swarm[i].sum()  # Re-normalize
            
            score, _, _ = train_multi_agent(swarm[i], max_episodes=eval_episodes,
                                           learning_rate=1e-3, gamma=0.99,
                                           hidden_size=128, entropy_coef=1e-3,
                                           beta_comm=0.1, headless=True,
                                           seed=seed, verbose=False)
            
            if verbose:
                print(f"  Particle {i+1}: Score={score:.2f}")
            
            if score > pbest_scores[i]:
                pbest_scores[i] = score
                pbest[i] = swarm[i].copy()
        
        best_idx = np.argmax(pbest_scores)
        if pbest_scores[best_idx] > gbest_score:
            gbest = pbest[best_idx].copy()
            gbest_score = pbest_scores[best_idx]
    
    if verbose:
        print(f"\nCOMM PSO DONE. Best weights: {gbest}, Score: {gbest_score:.2f}")
    
    return gbest, gbest_score

# -------------------- PSO for Hyperparameters --------------------
def short_eval_for_hyperparam_pso(hparams, comm_vector, eval_episodes=8, seed=None):
    log_lr = float(hparams.get('log_lr', -3.0))
    lr = 10.0 ** log_lr
    lr = float(np.clip(lr, 1e-6, 3e-3))
    gamma = float(hparams.get('gamma', 0.99))
    hidden = int(hparams.get('hidden_size', 128))
    entropy_coef = float(hparams.get('entropy_coef', 1e-3))
    beta_comm = float(hparams.get('beta_comm', 0.1))
    
    avg_return, _, _ = train_multi_agent(comm_vector, max_episodes=eval_episodes,
                                        learning_rate=lr, gamma=gamma,
                                        hidden_size=hidden, entropy_coef=entropy_coef,
                                        beta_comm=beta_comm, headless=True,
                                        seed=seed, verbose=False)
    return avg_return

def pso_optimize_hyperparams(comm_vector, param_bounds, n_particles=6, n_iters=6, 
                             w=0.5, c1=1.0, c2=1.0, seed=None, eval_episodes=8):
    """
    PSO for hyperparameters (given fixed comm_vector).
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    
    names = list(param_bounds.keys())
    dim = len(names)
    lower = np.array([param_bounds[n][0] for n in names], dtype=float)
    upper = np.array([param_bounds[n][1] for n in names], dtype=float)
    types = [param_bounds[n][2] for n in names]
    
    X = np.random.uniform(lower, upper, size=(n_particles, dim))
    V = np.zeros_like(X)
    pbest = X.copy()
    pbest_scores = np.full(n_particles, -np.inf)
    gbest = None
    gbest_score = -np.inf
    
    # Initial evaluation
    for i in range(n_particles):
        params = {}
        for j, name in enumerate(names):
            val = X[i,j]
            params[name] = int(round(val)) if types[j]=='int' else float(val)
        score = short_eval_for_hyperparam_pso(params, comm_vector, eval_episodes, seed)
        pbest_scores[i] = score
        if score > gbest_score:
            gbest_score = score
            gbest = X[i].copy()
    
    # Main PSO loop
    for _it in range(1, n_iters+1):
        for i in range(n_particles):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)
            V[i] = w * V[i] + c1 * r1 * (pbest[i] - X[i]) + c2 * r2 * (gbest - X[i])
            X[i] = X[i] + V[i]
            X[i] = np.minimum(np.maximum(X[i], lower), upper)
            
            params = {}
            for j, name in enumerate(names):
                val = X[i,j]
                params[name] = int(round(val)) if types[j]=='int' else float(val)
            
            score = short_eval_for_hyperparam_pso(params, comm_vector, eval_episodes, seed)
            if score > pbest_scores[i]:
                pbest_scores[i] = score
                pbest[i] = X[i].copy()
            if score > gbest_score:
                gbest_score = score
                gbest = X[i].copy()
    
    # Convert gbest to typed dict
    best_params = {}
    for j, name in enumerate(names):
        val = gbest[j]
        best_params[name] = int(round(val)) if types[j]=='int' else float(val)
    
    return best_params, gbest_score

# -------------------- Main Training Pipeline --------------------
def run_full_optimization():
    """
    Two-stage optimization:
    1. Find best communication weights using PSO
    2. Find best hyperparameters using PSO (with best comm weights)
    3. Train final model with both optimized
    """
    
    print("=" * 70)
    print("STAGE 1: Optimizing Communication Weights")
    print("=" * 70)
    
    # Stage 1: Optimize communication weights
    COMM_PSO_PARTICLES = 6
    COMM_PSO_ITERS = 5
    COMM_EVAL_EPISODES = 20
    
    best_comm_vector, best_comm_score = pso_optimize_comm(
        n_particles=COMM_PSO_PARTICLES,
        iters=COMM_PSO_ITERS,
        eval_episodes=COMM_EVAL_EPISODES,
        seed=123,
        verbose=True
    )
    
    print(f"\nBest communication weights: {best_comm_vector}")
    print(f"Best communication score: {best_comm_score:.3f}")
    
    # Save communication weights
    try:
        with open("best_comm_weights.json", "w") as f:
            json.dump({
                'weights': best_comm_vector.tolist(),
                'score': float(best_comm_score)
            }, f, indent=2)
        print("Communication weights saved to 'best_comm_weights.json'")
    except Exception as e:
        print(f"Could not save comm weights: {e}")
    
    print("\n" + "=" * 70)
    print("STAGE 2: Optimizing Hyperparameters (with best comm weights)")
    print("=" * 70)
    
    # Stage 2: Optimize hyperparameters using best comm weights
    param_bounds = {
        'log_lr': (-5.0, -2.0, 'float'),
        'gamma': (0.95, 0.995, 'float'),
        'hidden_size': (64, 192, 'int'),
        'entropy_coef': (0.0, 0.02, 'float'),
        'beta_comm': (0.05, 0.3, 'float'),  # Communication strength
    }
    
    HYPERPARAM_PSO_PARTICLES = 6
    HYPERPARAM_PSO_ITERS = 6
    HYPERPARAM_EVAL_EPISODES = 8
    
    print("\nStarting Hyperparameter PSO...")
    best_params, best_hyperparam_score = pso_optimize_hyperparams(
        comm_vector=best_comm_vector,
        param_bounds=param_bounds,
        n_particles=HYPERPARAM_PSO_PARTICLES,
        n_iters=HYPERPARAM_PSO_ITERS,
        w=0.6, c1=1.2, c2=1.2,
        seed=123,
        eval_episodes=HYPERPARAM_EVAL_EPISODES
    )
    
    print("\nBest hyperparameters (found by PSO):")
    for key, val in best_params.items():
        print(f"  {key}: {val}")
    
    # Save hyperparameters
    try:
        with open("best_hyperparameters.json", "w") as f:
            json.dump(best_params, f, indent=2)
        print("\nHyperparameters saved to 'best_hyperparameters.json'")
    except Exception:
        pass
    
    print("\n" + "=" * 70)
    print("STAGE 3: Final Training with Optimized Parameters")
    print("=" * 70)
    
    # Map best params
    lr = float(np.clip(10.0 ** best_params['log_lr'], 1e-6, 3e-3))
    gamma = float(best_params['gamma'])
    hidden = int(best_params['hidden_size'])
    entropy_coef = float(best_params['entropy_coef'])
    beta_comm = float(best_params['beta_comm'])
    
    print(f"\nTraining with:")
    print(f"  Learning Rate: {lr:.6f}")
    print(f"  Gamma: {gamma}")
    print(f"  Hidden Size: {hidden}")
    print(f"  Entropy Coef: {entropy_coef}")
    print(f"  Comm Beta: {beta_comm}")
    print(f"  Comm Weights: {best_comm_vector}")
    print()
    
    # Final training
    avg_return, success_rate, final_models = train_multi_agent(
        comm_vector=best_comm_vector,
        max_episodes=300,
        learning_rate=lr,
        gamma=gamma,
        hidden_size=hidden,
        entropy_coef=entropy_coef,
        beta_comm=beta_comm,
        print_interval=10,
        headless=True,
        clip_grad_norm=0.5,
        seed=123,
        verbose=True
    )
    
    # Save all agent models
    for i, model in enumerate(final_models):
        filename = f"actor_critic_agent_{i}_final.weights.h5"
        model.save_weights(filename)
        print(f"Agent {i} weights saved to '{filename}'")
    
    print("\n" + "=" * 70)
    print("TRAINING COMPLETE")
    print("=" * 70)
    print(f"Final Average Return: {avg_return:.3f}")
    print(f"Final Success Rate: {success_rate:.3f}")
    print(f"Communication Weights: {best_comm_vector}")
    print("=" * 70)

# -------------------- Run --------------------
if __name__ == "__main__":
    run_full_optimization()

STAGE 1: Optimizing Communication Weights

===== INITIAL COMM PSO EVALUATION =====
Particle 1: Score=109.94, Weights=[0.39554701 0.16250763 0.12883616 0.3131092 ]
Particle 2: Score=115.85, Weights=[0.25620569 0.15066985 0.34925393 0.24387053]
Particle 3: Score=81.32, Weights=[0.24723053 0.20157411 0.176416   0.37477935]
Particle 4: Score=76.80, Weights=[0.26835647 0.0365161  0.24355794 0.45156949]
Particle 5: Score=88.54, Weights=[0.12839573 0.12344261 0.37398366 0.374178  ]
Particle 6: Score=70.92, Weights=[0.22501981 0.3012905  0.25696178 0.21672791]

===== COMM PSO ITER 1/5 | Best=115.85 =====
  Particle 1: Score=81.56
  Particle 2: Score=81.89
  Particle 3: Score=108.51
  Particle 4: Score=57.88
  Particle 5: Score=72.10
  Particle 6: Score=106.93

===== COMM PSO ITER 2/5 | Best=115.85 =====
  Particle 1: Score=107.35
  Particle 2: Score=93.87
  Particle 3: Score=116.84
  Particle 4: Score=95.20
  Particle 5: Score=76.60
  Particle 6: Score=120.59

===== COMM PSO ITER 3/5 | Best=12

## Part 2: Inference on navigation scenarios
Loads the saved weights and runs three scenarios (*Cross the Divider*, *Parallel Navigation*, *Circular Formation*) in a Pygame window. Close the window to continue.

*Note: in the saved run, agents did not reliably complete the hardest scenario. See the README's "Results and limitations" section.*

In [8]:
import pygame
import numpy as np
import math
import json
import tensorflow as tf
from tensorflow.keras import layers

# Import all the classes and functions from your training script
# (Copy the relevant classes here)

pygame.init()

SCREEN_WIDTH = 1500
SCREEN_HEIGHT = 750
wall_thickness = 12

HEADLESS = False  # Set to False to visualize
MAX_STEPS = 500

ACTION_SIZE = 5
STATE_SIZE = 8
NUM_AGENTS = 4

SCREEN = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Multi-Agent Library Environment - Inference")

# Colors
WHITE = (255,255,255)
RED = (255,0,0)
table_color = (100,255,100)
chair_color = (150,75,0)
shelf_color = (139,69,19)
bg_color = (210,210,210)
wall_color = (100,100,100)
robot_colors = [(50,100,150), (150,50,100), (100,150,50), (150,100,50)]
cleaner_color = (230,0,0)

# Static obstacles
static_obstacles = {
    'walls': [
        pygame.Rect(0, 50, SCREEN_WIDTH, wall_thickness),
        pygame.Rect(0, 738, SCREEN_WIDTH, wall_thickness),
        pygame.Rect(0, 50, 12, 700),
        pygame.Rect(1488, 50, 12, 700),
        pygame.Rect(750, 62, 12, 480),
    ],
    'shelves': [
        pygame.Rect(232, 200, 20, 250),
        pygame.Rect(482, 200, 20, 250),
    ],
    'tables': [
        pygame.Rect(972, 200, 75, 40),
        pygame.Rect(1212, 200, 75, 40),
        pygame.Rect(972, 400, 75, 40),
        pygame.Rect(1212, 400, 75, 40),
    ],
    'chairs': [
        pygame.Rect(987, 180, 15, 15),
        pygame.Rect(1017, 180, 15, 15),
        pygame.Rect(987, 245, 15, 15),
        pygame.Rect(1017, 245, 15, 15),
        pygame.Rect(1227, 180, 15, 15),
        pygame.Rect(1257, 180, 15, 15),
        pygame.Rect(1227, 245, 15, 15),
        pygame.Rect(1257, 245, 15, 15),
        pygame.Rect(987, 380, 15, 15),
        pygame.Rect(1017, 380, 15, 15),
        pygame.Rect(987, 445, 15, 15),
        pygame.Rect(1017, 445, 15, 15),
        pygame.Rect(1227, 380, 15, 15),
        pygame.Rect(1257, 380, 15, 15),
        pygame.Rect(1227, 445, 15, 15),
        pygame.Rect(1257, 445, 15, 15),
    ]
}

def check_collision_with_static_obstacles(rect):
    for obstacles_group in static_obstacles.values():
        for obstacle in obstacles_group:
            if rect.colliderect(obstacle):
                return True
    return False

class Robot:
    num = 0
    width = 20
    height = 50
    max_velocity = 8.0
    min_velocity = 1.0
    all_robots = []

    def __init__(self, position, theta, robot_id):
        Robot.num += 1
        self.num = Robot.num
        self.robot_id = robot_id
        self.position = np.array(position, dtype=float)
        self.theta = float(theta)
        self.rect = pygame.Rect(int(self.position[0]), int(self.position[1]), Robot.width, Robot.height)
        self.current_velocity = 3.0
        Robot.all_robots.append(self)

    def check_collision_with_other_robots(self, rect):
        for robot in Robot.all_robots:
            if robot.num == self.num:
                continue
            if rect.colliderect(robot.rect):
                return True
        return False

    def apply_action(self, action):
        rot_step = 0.08
        accel = 0.6
        if action == 0:
            self.current_velocity = min(self.current_velocity + accel, Robot.max_velocity)
        elif action == 1:
            self.current_velocity = max(self.current_velocity - accel, -Robot.max_velocity/2.0)
        elif action == 2:
            self.theta = (self.theta - rot_step) % (2*math.pi)
        elif action == 3:
            self.theta = (self.theta + rot_step) % (2*math.pi)
        elif action == 4:
            self.current_velocity *= 0.9

    def update_physics(self):
        future_position = np.zeros(2, dtype=float)
        future_position[0] = self.position[0] + self.current_velocity * math.cos(self.theta)
        future_position[1] = self.position[1] + self.current_velocity * math.sin(self.theta)
        future_rect = pygame.Rect(int(future_position[0]), int(future_position[1]), Robot.width, Robot.height)
        if check_collision_with_static_obstacles(future_rect) or self.check_collision_with_other_robots(future_rect):
            self.theta = (self.theta + math.pi/10) % (2*math.pi)
            self.current_velocity = 0.0
        else:
            self.position = future_position
            self.rect = future_rect

class Cleaner:
    count_cleaners = 0
    max_velocity = 4
    min_velocity = 2
    width = 20
    height = 20
    all_cleaners = []

    def __init__(self, position, theta):
        Cleaner.count_cleaners += 1
        self.num = Cleaner.count_cleaners
        self.position = np.array(position, dtype=float)
        self.theta = float(theta)
        self.current_velocity = Cleaner.min_velocity
        self.rect = pygame.Rect(int(position[0]), int(position[1]), Cleaner.width, Cleaner.height)
        Cleaner.all_cleaners.append(self)

    def check_collision_cleaner(self, future_rect):
        for cleaner in Cleaner.all_cleaners:
            if cleaner.num == self.num:
                continue
            if future_rect.colliderect(cleaner.rect):
                return True
        return False

    def check_collision_robot(self, future_rect):
        for robot in Robot.all_robots:
            if future_rect.colliderect(robot.rect):
                return True
        return False

    def move_cleaner(self):
        future_position = np.zeros(2, dtype=float)
        future_position[0] = self.position[0] + self.current_velocity * math.cos(self.theta)
        future_position[1] = self.position[1] + self.current_velocity * math.sin(self.theta)
        future_rect = pygame.Rect(int(future_position[0]), int(future_position[1]), Cleaner.width, Cleaner.height)
        if (check_collision_with_static_obstacles(future_rect)
            or self.check_collision_cleaner(future_rect)
            or self.check_collision_robot(future_rect)):
            self.theta = (self.theta + math.pi) % (2*math.pi)
        else:
            self.position = future_position
            self.rect = future_rect
            self.current_velocity = Cleaner.min_velocity if self.position[0] >= 750 else Cleaner.max_velocity

def draw_environment(screen):
    screen.fill(bg_color)
    font_color = (0, 0, 0)
    font = pygame.font.SysFont('Arial', 26, bold=True)
    screen.blit(font.render("Bookshelves Room", True, font_color), (wall_thickness + 245, 15))
    screen.blit(font.render("Quiet Area", True, font_color), (500 + wall_thickness + 540, 15))
    
    for obstacles_group in static_obstacles.keys():
        if obstacles_group == 'walls':
            obstacle_color = wall_color
        elif obstacles_group == 'shelves':
            obstacle_color = shelf_color
        elif obstacles_group == 'tables':
            obstacle_color = table_color
        elif obstacles_group == 'chairs':
            obstacle_color = chair_color
        for obstacle in static_obstacles[obstacles_group]:
            pygame.draw.rect(screen, obstacle_color, obstacle)

    # Draw goal positions and start positions for each robot
    for robot in Robot.all_robots:
        color = robot_colors[robot.robot_id % len(robot_colors)]
        
        # Draw start position (lighter version)
        start_color = tuple(min(255, c + 100) for c in color)
        pygame.draw.circle(screen, start_color, 
                          (int(robot.start_position[0]), int(robot.start_position[1])), 15, 2)
        
        # Draw goal position (circle with 'G')
        goal_x, goal_y = int(robot.goal_position[0]), int(robot.goal_position[1])
        pygame.draw.circle(screen, color, (goal_x, goal_y), 25, 3)
        goal_font = pygame.font.SysFont('Arial', 20, bold=True)
        goal_text = goal_font.render('G', True, color)
        screen.blit(goal_text, (goal_x - 7, goal_y - 10))
        
        # Draw path line from start to goal
        pygame.draw.line(screen, color, 
                        (int(robot.start_position[0]), int(robot.start_position[1])),
                        (goal_x, goal_y), 1)
        
        # Draw robot
        pygame.draw.rect(screen, color, robot.rect)
        
        # Draw direction indicator
        end_x = int(robot.position[0] + 30 * math.cos(robot.theta))
        end_y = int(robot.position[1] + 30 * math.sin(robot.theta))
        pygame.draw.line(screen, WHITE, 
                        (int(robot.position[0] + Robot.width//2), 
                         int(robot.position[1] + Robot.height//2)),
                        (end_x, end_y), 2)

    for cleaner in Cleaner.all_cleaners:
        pygame.draw.rect(screen, cleaner_color, cleaner.rect)

def env_reset(start_positions=None, goal_positions=None):
    """
    Reset environment with custom start and goal positions for each agent.
    start_positions: list of [x, y] for each agent
    goal_positions: list of [x, y] for each agent (stored as agent attribute)
    """
    Robot.all_robots = []
    Cleaner.all_cleaners = []
    Robot.num = 0
    Cleaner.count_cleaners = 0
    
    # Default positions if none provided
    if start_positions is None:
        start_positions = [
            [100, 500],   # Agent 0
            [600, 150],   # Agent 1
            [880, 300],   # Agent 2
            [1400, 600]   # Agent 3
        ]
    
    if goal_positions is None:
        goal_positions = [
            [750, 500],   # Agent 0 goal
            [750, 150],   # Agent 1 goal
            [750, 300],   # Agent 2 goal
            [750, 600]    # Agent 3 goal
        ]
    
    # Create robots with start positions and assign goals
    for i in range(NUM_AGENTS):
        robot = Robot(np.array(start_positions[i]), 0, robot_id=i)
        robot.goal_position = np.array(goal_positions[i], dtype=float)
        robot.start_position = np.array(start_positions[i], dtype=float)
    
    Cleaner(np.array([200,500]), 0)
    Cleaner(np.array([900,680]), 0)
    
    return [env_get_obs(i).astype(np.float32) for i in range(NUM_AGENTS)]

def env_get_obs(robot_idx=0):
    """Get observation for a robot - now includes goal direction"""
    robot = Robot.all_robots[robot_idx]
    x_norm = (robot.position[0] / SCREEN_WIDTH) * 2 - 1
    y_norm = (robot.position[1] / SCREEN_HEIGHT) * 2 - 1
    cth = math.cos(robot.theta)
    sth = math.sin(robot.theta)
    
    # Direction to goal instead of cleaner
    if hasattr(robot, 'goal_position'):
        dx = (robot.goal_position[0] - robot.position[0]) / SCREEN_WIDTH
        dy = (robot.goal_position[1] - robot.position[1]) / SCREEN_HEIGHT
    else:
        # Fallback to cleaner direction if no goal
        if Cleaner.all_cleaners:
            dists = [np.linalg.norm(cl.position - robot.position) for cl in Cleaner.all_cleaners]
            idx = int(np.argmin(dists))
            dx = (Cleaner.all_cleaners[idx].position[0] - robot.position[0]) / SCREEN_WIDTH
            dy = (Cleaner.all_cleaners[idx].position[1] - robot.position[1]) / SCREEN_HEIGHT
        else:
            dx, dy = 0.0, 0.0
    
    min_dist = 1e9
    for group in static_obstacles.values():
        for obs in group:
            center = np.array([obs.centerx, obs.centery], dtype=float)
            d = np.linalg.norm(center - robot.position)
            if d < min_dist:
                min_dist = d
    
    if min_dist == 1e9:
        min_dist_norm = 1.0
    else:
        max_diag = math.hypot(SCREEN_WIDTH, SCREEN_HEIGHT)
        min_dist_norm = np.clip(min_dist / max_diag, 0.0, 1.0)
        min_dist_norm = min_dist_norm * 2 - 1
    
    vel_norm = (robot.current_velocity - Robot.min_velocity) / max(1e-8, (Robot.max_velocity - Robot.min_velocity))
    vel_norm = vel_norm * 2 - 1
    
    obs = np.array([x_norm, y_norm, cth, sth, dx, dy, min_dist_norm, vel_norm], dtype=np.float32)
    return np.clip(obs, -1.0, 1.0)

class ActorCritic(tf.keras.Model):
    def __init__(self, state_size, action_size, hidden_size=128):
        super(ActorCritic, self).__init__()
        self.fc1 = layers.Dense(hidden_size, activation='relu')
        self.actor = layers.Dense(action_size, activation='softmax')
        self.critic = layers.Dense(1)
    
    def call(self, x):
        x = self.fc1(x)
        return self.actor(x), self.critic(x)

def load_models(hidden_size=128):
    """Load all trained agent models"""
    models = []
    for i in range(NUM_AGENTS):
        model = ActorCritic(STATE_SIZE, ACTION_SIZE, hidden_size)
        model(tf.zeros((1, STATE_SIZE), dtype=tf.float32))  # Build the model
        
        try:
            model.load_weights(f"actor_critic_agent_{i}_final.weights.h5")
            print(f"✓ Loaded Agent {i} weights")
            models.append(model)
        except Exception as e:
            print(f"✗ Could not load Agent {i} weights: {e}")
            return None
    
    return models

def run_inference(models, tasks, render_delay=30):
    """
    Run inference with loaded models on specified tasks.
    tasks: list of dicts with 'name', 'starts', 'goals', 'description'
    """
    
    running = True
    
    for task_idx, task in enumerate(tasks):
        if not running:
            break
            
        print(f"\n{'='*60}")
        print(f"TASK {task_idx + 1}: {task['name']}")
        print(f"Description: {task['description']}")
        print('='*60)
        
        # Show task details
        for i in range(NUM_AGENTS):
            start = task['starts'][i]
            goal = task['goals'][i]
            dist = math.hypot(goal[0] - start[0], goal[1] - start[1])
            print(f"  Agent {i}: ({start[0]:.0f}, {start[1]:.0f}) → ({goal[0]:.0f}, {goal[1]:.0f})  [Distance: {dist:.0f}px]")
        
        states = env_reset(start_positions=task['starts'], goal_positions=task['goals'])
        dones = [False] * NUM_AGENTS
        total_rewards = [0.0] * NUM_AGENTS
        steps = 0
        goal_threshold = 30  # Distance to consider goal reached
        
        clock = pygame.time.Clock()
        
        while not all(dones) and steps < MAX_STEPS and running:
            # Handle pygame events
            try:
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        running = False
                        break
            except:
                # If pygame fails, stop running
                running = False
                break
            
            if not running:
                break
            
            # Get actions from trained models
            actions = []
            for i in range(NUM_AGENTS):
                if not dones[i]:
                    state_tensor = tf.convert_to_tensor([states[i]], dtype=tf.float32)
                    probs, _ = models[i](state_tensor)
                    action = np.argmax(probs.numpy()[0])  # Greedy action
                    actions.append(action)
                else:
                    actions.append(4)  # No-op
            
            # Step environment
            next_states, rewards, new_dones, _ = env_step(actions, steps)
            
            # Update tracking
            for i in range(NUM_AGENTS):
                if not dones[i]:
                    total_rewards[i] += rewards[i]
                    
                    # Check if goal reached
                    robot = Robot.all_robots[i]
                    dist_to_goal = np.linalg.norm(robot.position - robot.goal_position)
                    
                    if dist_to_goal < goal_threshold or new_dones[i]:
                        dones[i] = True
                        success = dist_to_goal < goal_threshold
                        status = "✓ GOAL REACHED!" if success else "✗ Failed"
                        print(f"  Agent {i} @ Step {steps}: {status} (Reward: {total_rewards[i]:.2f}, Final Dist: {dist_to_goal:.0f}px)")
            
            states = next_states
            steps += 1
            
            # Render
            try:
                draw_environment(SCREEN)
                
                # Display task info
                font_large = pygame.font.SysFont('Arial', 22, bold=True)
                font_small = pygame.font.SysFont('Arial', 18)
                
                task_text = font_large.render(f"TASK: {task['name']}", True, (0, 0, 0))
                SCREEN.blit(task_text, (20, 650))
                
                info_y = 680
                for i in range(NUM_AGENTS):
                    robot = Robot.all_robots[i]
                    dist_to_goal = np.linalg.norm(robot.position - robot.goal_position)
                    status = "DONE" if dones[i] else f"Moving (Dist: {dist_to_goal:.0f}px)"
                    color = (0, 200, 0) if dones[i] else robot_colors[i]
                    text = font_small.render(f"Agent {i}: {status} | Reward: {total_rewards[i]:.1f}", 
                                     True, color)
                    SCREEN.blit(text, (20, info_y + i * 20))
                
                pygame.display.flip()
                clock.tick(render_delay)  # FPS control
            except:
                # If rendering fails, continue without visualization
                running = False
                break
        
        if not running:
            print("\n  Task interrupted by user")
            break
        
        # Task summary
        print(f"\n  Task Summary:")
        print(f"    Total Steps: {steps}")
        print(f"    Average Reward: {np.mean(total_rewards):.2f}")
        
        successful = 0
        for i in range(NUM_AGENTS):
            robot = Robot.all_robots[i]
            dist_to_goal = np.linalg.norm(robot.position - robot.goal_position)
            if dist_to_goal < goal_threshold:
                successful += 1
        
        print(f"    Success Rate: {successful}/{NUM_AGENTS} agents ({successful/NUM_AGENTS*100:.0f}%)")
        
        # Wait before next task
        if task_idx < len(tasks) - 1 and running:
            print("\n  [Next task in 2 seconds...]")
            try:
                pygame.time.wait(2000)
            except:
                pass

def env_step(actions, current_step):
    """Step the environment with goal-based rewards"""
    observations = []
    rewards = []
    dones = []
    infos = []
    
    for i, robot in enumerate(Robot.all_robots):
        # Calculate distance to goal
        prev_dist = np.linalg.norm(robot.goal_position - robot.position)
        
        robot.apply_action(actions[i])
        robot.update_physics()
        
        new_dist = np.linalg.norm(robot.goal_position - robot.position)
        
        # Reward shaping: encourage moving toward goal
        shaping_reward = 0.5 * (prev_dist - new_dist)
        step_penalty = -0.001
        reward = shaping_reward + step_penalty
        done = False
        
        # Collision penalties
        if robot.check_collision_with_other_robots(robot.rect) or check_collision_with_static_obstacles(robot.rect):
            reward = -1.0
            done = True
        
        for cl in Cleaner.all_cleaners:
            if robot.rect.colliderect(cl.rect):
                reward = -1.0
                done = True
        
        # Goal reached reward
        if new_dist < 30:  # Within 30 pixels of goal
            reward = 10.0
            done = True
        
        if current_step >= MAX_STEPS:
            done = True
        
        rewards.append(float(reward))
        dones.append(bool(done))
        infos.append({})
    
    for cl in Cleaner.all_cleaners:
        cl.move_cleaner()
    
    for i in range(NUM_AGENTS):
        observations.append(env_get_obs(i).astype(np.float32))
    
    return observations, rewards, dones, infos

def main():
    print("="*60)
    print("Multi-Agent Point-to-Point Navigation Tasks")
    print("="*60)
    
    # Initialize pygame properly
    try:
        pygame.init()
        pygame.display.init()
    except Exception as e:
        print(f"Error initializing pygame: {e}")
        return
    
    # Try to load hyperparameters
    hidden_size = 128
    try:
        with open("best_hyperparameters.json", "r") as f:
            params = json.load(f)
            hidden_size = int(params.get('hidden_size', 128))
            print(f"\nLoaded hyperparameters: hidden_size={hidden_size}")
    except:
        print(f"\nUsing default hidden_size={hidden_size}")
    
    # Try to load communication weights
    try:
        with open("best_comm_weights.json", "r") as f:
            comm_data = json.load(f)
            print(f"Communication weights: {comm_data['weights']}")
            print(f"Communication score: {comm_data['score']:.2f}")
    except:
        print("No communication weights file found")
    
    # Load models
    print("\nLoading trained models...")
    models = load_models(hidden_size)
    
    if models is None:
        print("\n✗ Failed to load models. Please ensure weight files exist:")
        for i in range(NUM_AGENTS):
            print(f"  - actor_critic_agent_{i}_final.weights.h5")
        pygame.quit()
        return
    
    print(f"\n✓ Successfully loaded all {NUM_AGENTS} agent models!")
    
    # Define navigation tasks
    tasks = [
        {
            'name': 'Cross the Divider',
            'description': 'All agents move from left room to right room',
            'starts': [
                [100, 500],   # Agent 0: bottom-left
                [600, 150],   # Agent 1: top-left
                [300, 300],   # Agent 2: mid-left
                [400, 600],   # Agent 3: bottom-left
            ],
            'goals': [
                [900, 500],   # Agent 0: bottom-right
                [1200, 150],  # Agent 1: top-right
                [1000, 300],  # Agent 2: mid-right
                [1300, 600],  # Agent 3: bottom-right
            ]
        },
        {
            'name': 'Parallel Navigation',
            'description': 'Agents move in parallel lanes',
            'starts': [
                [100, 150],   # Agent 0: lane 1
                [100, 300],   # Agent 1: lane 2
                [100, 450],   # Agent 2: lane 3
                [100, 600],   # Agent 3: lane 4
            ],
            'goals': [
                [1400, 150],  # Agent 0: same lane
                [1400, 300],  # Agent 1: same lane
                [1400, 450],  # Agent 2: same lane
                [1400, 600],  # Agent 3: same lane
            ]
        },
        {
            'name': 'Circular Formation',
            'description': 'Agents form a circle around tables area',
            'starts': [
                [800, 150],   # Agent 0: top
                [1350, 350],  # Agent 1: right
                [800, 600],   # Agent 2: bottom
                [850, 350],   # Agent 3: left
            ],
            'goals': [
                [1100, 250],  # Agent 0
                [1200, 350],  # Agent 1
                [1100, 450],  # Agent 2
                [1000, 350],  # Agent 3
            ]
        }
    ]
    
    print("\n" + "="*60)
    print(f"Loaded {len(tasks)} navigation tasks")
    print("="*60)
    
    # Run tasks
    print("\nStarting tasks (close window to exit)...")
    print("Legend: Circle with 'G' = Goal, Light circle = Start position\n")
    print("Visualization: Watch each agent navigate from start to goal!")
    print("- Light circles show where each agent started")
    print("- 'G' markers show goal positions")
    print("- White arrows show agent direction\n")
    
    try:
        run_inference(models, tasks, render_delay=30)
    except KeyboardInterrupt:
        print("\nStopped by user")
    except Exception as e:
        print(f"\nError during inference: {e}")
    finally:
        try:
            pygame.quit()
        except:
            pass
        print("\n" + "="*60)
        print("All tasks complete!")
        print("="*60)

if __name__ == "__main__":
    main()

if __name__ == "__main__":
    main()

Multi-Agent Point-to-Point Navigation Tasks

Loaded hyperparameters: hidden_size=95
Communication weights: [0.23274387282965991, 0.07303847385511808, 0.3992060525391356, 0.29501160077608646]
Communication score: 125.09

Loading trained models...
✓ Loaded Agent 0 weights
✓ Loaded Agent 1 weights
✓ Loaded Agent 2 weights
✓ Loaded Agent 3 weights

✓ Successfully loaded all 4 agent models!

Loaded 3 navigation tasks

Starting tasks (close window to exit)...
Legend: Circle with 'G' = Goal, Light circle = Start position

Visualization: Watch each agent navigate from start to goal!
- Light circles show where each agent started
- 'G' markers show goal positions
- White arrows show agent direction


TASK 1: Cross the Divider
Description: All agents move from left room to right room
  Agent 0: (100, 500) → (900, 500)  [Distance: 800px]
  Agent 1: (600, 150) → (1200, 150)  [Distance: 600px]
  Agent 2: (300, 300) → (1000, 300)  [Distance: 700px]
  Agent 3: (400, 600) → (1300, 600)  [Distance: 900p